# Solutions – Day 28

In [ ]:
# Exercise 1: Persist long‑term memory
import json

def save_memory(mem, path="long_mem.json"):
    with open(path, "w") as f:
        json.dump({"facts": mem.facts}, f)

def load_memory(mem, path="long_mem.json"):
    if os.path.exists(path):
        with open(path, "r") as f:
            data = json.load(f)
            mem.facts = data["facts"]
            mem._rebuild_index()

# Usage
# save_memory(long_mem)
# load_memory(long_mem)

In [ ]:
# Exercise 2: Automatic fact extraction
def extract_facts(user_message: str) -> List[str]:
    prompt = f"Extract personal facts from this message. Return as bullet points: {user_message}"
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}]
    )
    content = response.choices[0].message.content
    facts = [line.strip("- ") for line in content.split("\n") if line.startswith("-")]
    return facts

# Add to ask_with_memory: for each user message, call extract_facts and add to long_mem

In [ ]:
# Exercise 3: Forgetting mechanism
class ForgettingLongTermMemory(LongTermMemory):
    def __init__(self, max_facts=20):
        super().__init__()
        self.max_facts = max_facts
        self.access_count = {}
    
    def add_fact(self, fact):
        super().add_fact(fact)
        self.access_count[fact] = 0
        if len(self.facts) > self.max_facts:
            # remove least accessed
            least = min(self.access_count, key=self.access_count.get)
            idx = self.facts.index(least)
            self.facts.pop(idx)
            del self.access_count[least]
            self._rebuild_index()
    
    def retrieve(self, query, k=2):
        results = super().retrieve(query, k)
        for r in results:
            self.access_count[r] = self.access_count.get(r, 0) + 1
        return results

In [ ]:
# Exercise 4: Hybrid retrieval
# Pseudocode: get recent short‑term messages (recency = 1/(index+1)),
# get long‑term facts with similarity scores, combine with weighted sum.

In [ ]:
# Exercise 5: LangGraph integration
# Add memory to the state: state["short_mem"] and state["long_mem"].
# In agent_node, retrieve relevant facts and add to system message.